In [1]:
# 🔌 Cell 1 & 2 - 引擎初始化与多密钥自动轮转池 (Infrastructure & Quota Pooling)
# -----------------------------------------------------------
import os
import yaml
import time
import isodate
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from pyprojroot import here
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# 1. 挂载绝对路径与环境
PROJECT_ROOT = here()
DATA_DIR = PROJECT_ROOT / "Data"
CONFIG_DIR = PROJECT_ROOT / "configs"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = DATA_DIR / f"data_01_video_candidates_{timestamp}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🌟 加载多密钥阵列 (破除单一 Quota 瓶颈)
env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

def load_all_youtube_api_keys():
    keys = []
    i = 1
    while True:
        key = os.getenv(f"YOUTUBE_API_KEY_{i}")
        if key:
            keys.append(key)
            i += 1
        else:
            break
    return keys

api_keys = load_all_youtube_api_keys()

if not api_keys:
    raise ValueError(f"❌ 无法在 {env_path} 找到任何 YOUTUBE_API_KEY_x 变量，请检查配置！")

print(f"✅ 成功加载 {len(api_keys)} 个 YouTube API 密钥！资产将落地至: {OUTPUT_DIR.name}")

# 2. 建立带状态的 API 客户端引擎
current_key_idx = 0

def get_youtube_client(key_index):
    return build("youtube", "v3", developerKey=api_keys[key_index])

# 初始化全局客户端
youtube = get_youtube_client(current_key_idx)

# 🌟 工业级核心：指数退避 + 自动密钥轮转
def api_request_with_retry(request_func, max_retries=5, base_delay=2):
    global current_key_idx, youtube  # 允许修改全局通信管道
    
    for attempt in range(max_retries):
        try:
            return request_func()
        except HttpError as e:
            # 403: Quota 超限; 429: 请求过快
            if e.resp.status in [403, 429]:
                print(f"  ⚠️ 触发 API 频控或配额耗尽 (状态码 {e.resp.status}).")
                
                # 如果有备用密钥，立即执行无缝轮转切换
                if len(api_keys) > 1:
                    current_key_idx = (current_key_idx + 1) % len(api_keys)
                    print(f"  🔄 正在自动轮转至备用密钥 [YOUTUBE_API_KEY_{current_key_idx + 1}]...")
                    youtube = get_youtube_client(current_key_idx)
                    time.sleep(1)  # 切换后微小停顿，避免并发冲突
                    continue       # 使用新密钥立即重试！
                
                # 如果只有 1 个密钥或全部耗尽，执行指数退避
                sleep_time = base_delay * (2 ** attempt)
                print(f"  ⏳ 无可用备用密钥，{sleep_time} 秒后退避重试...")
                time.sleep(sleep_time)
            else:
                raise e
    raise Exception("❌ API 请求失败次数超限，Pipeline 熔断！")

✅ 成功加载 7 个 YouTube API 密钥！资产将落地至: data_01_video_candidates_20260226_210856


In [2]:
# 📜 Cell 3 - 战略检索配置装载 (Strategic Configuration Loading)
# -----------------------------------------------------------
# 读取 2x2 矩阵与全局 API 配置 (解耦代码与业务逻辑)
config_path = CONFIG_DIR / "youtube_search.yaml" # 确保这里是你的真实文件名

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 🌟 1. 提取全局 API 门禁设定 (完美衔接你的 YAML 设计)
api_settings = config.get("youtube_api", {})
print("⚙️ 全局 API 门禁设定加载完毕:")
print(f"  - 排序准则: {api_settings.get('order', 'relevance')}")
print(f"  - 时间截断: {api_settings.get('published_after', 'None')}")
print(f"  - 物理底线: 时长 >= {api_settings.get('min_duration_minutes', 3)} 分钟 | 评论数 >= {api_settings.get('min_comment_count', 200)}")

# 🌟 2. 提取并降维 2x2 矩阵的 Keywords
groups_config = config.get("groups", {})

# 将多层级的字典扁平化为下游爬虫需要的结构：{ "cell_id": ["query1", "query2"] }
SEARCH_QUERIES = {}
for cell_id, cell_data in groups_config.items():
    SEARCH_QUERIES[cell_id] = cell_data.get("keywords", [])

if len(SEARCH_QUERIES) != 4:
    print("\n⚠️ 警告：检测到的分类不是标准的 4 组 2x2 矩阵，请检查 yaml 配置文件！")
else:
    print("\n🎯 2x2 战略寻址阵营解析成功:")
    for cell_id, queries in SEARCH_QUERIES.items():
        desc = groups_config[cell_id].get("description", "")
        print(f"  ✅ [{cell_id}] -> 包含 {len(queries)} 个靶向词 | 阵营定义: {desc}")

⚙️ 全局 API 门禁设定加载完毕:
  - 排序准则: relevance
  - 时间截断: 2020-01-01T00:00:00Z
  - 物理底线: 时长 >= 2 分钟 | 评论数 >= 50

🎯 2x2 战略寻址阵营解析成功:
  ✅ [shein_labor] -> 包含 6 个靶向词 | 阵营定义: SHEIN × LABOR: modern slavery / forced labor / sweatshop etc.
  ✅ [shein_environment] -> 包含 7 个靶向词 | 阵营定义: SHEIN × ENV: pollution / waste / sustainability / greenwashing etc.
  ✅ [non_shein_labor] -> 包含 9 个靶向词 | 阵营定义: NON_SHEIN × LABOR: other fast fashion brands labor issues.
  ✅ [non_shein_environment] -> 包含 7 个靶向词 | 阵营定义: NON_SHEIN × ENV: other fast fashion brands environmental issues.


In [3]:
# 🕷️ Cell 4 & 5 - 核心寻址引擎定义 (Search Engine Definition)
# -----------------------------------------------------------

def search_videos(youtube_client, query, max_results=50):
    """
    根据关键词进行靶向检索。强制限制类型为 video，语言为 en。
    每次请求耗费 100 Quota。
    """
    def _request():
        request = youtube_client.search().list(
            part="id",
            q=query,
            type="video",               # 🛡️ 门禁：只抓取视频，排除频道和播放列表
            relevanceLanguage="en",     # 🛡️ 门禁：锁定英语语料
            maxResults=max_results,
            order="relevance"           # 默认按相关性排序
        )
        return request.execute()
    
    response = api_request_with_retry(_request)
    return [item["id"]["videoId"] for item in response.get("items", [])]

def get_video_stats(youtube_client, video_ids):
    """
    批量富化视频元数据（最高 50 个 ID 一批）。
    每次批量请求耗费 1 Quota。
    """
    if not video_ids: return []
    
    stats = []
    # YouTube API 限制：一次最多查询 50 个视频详情
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        def _request():
            request = youtube_client.videos().list(
                part="snippet,statistics,contentDetails,status",
                id=",".join(batch_ids)
            )
            return request.execute()
            
        response = api_request_with_retry(_request)
        stats.extend(response.get("items", []))
        
    return stats

In [4]:
# 🚀 Cell 6 & 7 - 自动化检索流水线与元数据富化 (Data Lineage & Enrichment)
# -----------------------------------------------------------
from tqdm.auto import tqdm

all_videos_data = []

# 遍历 2x2 矩阵配置
for cell_id, queries in SEARCH_QUERIES.items():
    print(f"\n🚀 正在战略寻址阵营 [ {cell_id} ] ...")
    for query in tqdm(queries, desc=f"Querying {cell_id}"):
        
        # 分页获取基础视频 ID
        video_ids = search_videos(youtube, query, max_results=50)
        if not video_ids: continue
            
        # 批量富化元数据 (耗费极少 Quota)
        stats = get_video_stats(youtube, video_ids)
        
        for stat in stats:
            try:
                # 解析时长 (ISO 8601 转为分钟)
                duration_td = isodate.parse_duration(stat['contentDetails']['duration'])
                duration_min = duration_td.total_seconds() / 60.0
                
                # 提取互动数据与状态
                comment_count = int(stat['statistics'].get('commentCount', 0))
                privacy_status = stat['status'].get('privacyStatus', '')
                
                # 🛡️ 门禁 1: 物理拦截 Shorts (< 3分钟)
                # 🛡️ 门禁 2: 拦截互动死水 (< 100 评论)
                # 🛡️ 门禁 3: 确保视频公开可用
                if duration_min >= 3.0 and comment_count >= 100 and privacy_status == 'public':
                    all_videos_data.append({
                        "video_id": stat['id'],
                        "cell_id": cell_id,              # 🌟 核心血缘：记录所属 2x2 阵营
                        "source_query": query,           # 🌟 核心血缘：记录触发检索的词
                        "viewCount": int(stat['statistics'].get('viewCount', 0)),
                        "likeCount": int(stat['statistics'].get('likeCount', 0)),
                        "commentCount": comment_count,
                        "duration_min": round(duration_min, 2)
                    })
            except Exception as e:
                pass # 忽略部分字段缺失的残次品视频

df_candidates = pd.DataFrame(all_videos_data)
print(f"\n✅ 原始候选池构建完毕，通过硬性门禁的候选人总计: {len(df_candidates)} 条。")


🚀 正在战略寻址阵营 [ shein_labor ] ...


Querying shein_labor:   0%|          | 0/6 [00:00<?, ?it/s]


🚀 正在战略寻址阵营 [ shein_environment ] ...


Querying shein_environment:   0%|          | 0/7 [00:00<?, ?it/s]


🚀 正在战略寻址阵营 [ non_shein_labor ] ...


Querying non_shein_labor:   0%|          | 0/9 [00:00<?, ?it/s]


🚀 正在战略寻址阵营 [ non_shein_environment ] ...


Querying non_shein_environment:   0%|          | 0/7 [00:00<?, ?it/s]


✅ 原始候选池构建完毕，通过硬性门禁的候选人总计: 266 条。


In [5]:
# 🚨 Cell 8 - 交叉污染阻断与人工干预机制 (Contamination Handling)
# -----------------------------------------------------------
print("="*50)
print("🚨 执行交叉污染校验 (Cross-Cell Contamination Check)")
print("="*50)

# 1. 查找重复命中多个阵营的视频 (破坏独立同分布假设的元凶)
video_cell_counts = df_candidates.groupby('video_id')['cell_id'].nunique()
conflict_video_ids = video_cell_counts[video_cell_counts > 1].index.tolist()

if conflict_video_ids:
    print(f"⚠️ 警告: 探测到 {len(conflict_video_ids)} 个视频跨越了 2x2 边界！")
    
    # 隔离冲突资产至 CSV，供后续 Human-in-the-loop 人工审查
    df_conflicts = df_candidates[df_candidates['video_id'].isin(conflict_video_ids)]
    conflict_path = OUTPUT_DIR / "01_video_conflicts_for_human_review.csv"
    df_conflicts.to_csv(conflict_path, index=False, encoding='utf-8-sig')
    print(f"📂 冲突视频已隔离至: {conflict_path.name}")
    
    # 强制从主干管线中剔除这些污染源
    df_pure = df_candidates[~df_candidates['video_id'].isin(conflict_video_ids)].copy()
else:
    print("✅ 未探测到跨阵营污染，组间独立性完好！")
    df_pure = df_candidates.copy()

# 2. 内部去重 (同一个 query 或同 cell 内的不同 query 抓到了同一个视频)
df_pure = df_pure.drop_duplicates(subset=['video_id', 'cell_id'], keep='first')
print(f"📉 纯净候选池剩余靶向视频: {len(df_pure)} 条。")

🚨 执行交叉污染校验 (Cross-Cell Contamination Check)
⚠️ 警告: 探测到 25 个视频跨越了 2x2 边界！
📂 冲突视频已隔离至: 01_video_conflicts_for_human_review.csv
📉 纯净候选池剩余靶向视频: 78 条。


In [6]:
# ⚖️ Cell 9 - 仅执行底噪过滤，保留真实顶流生态 (Preserving Virality)
# -----------------------------------------------------------
print("\n" + "="*50)
print("⚖️ 执行底噪过滤 (Removing Dead-water Videos Only)")
print("="*50)

# 🌟 听从高级统计学设计的建议：绝不砍掉顶流爆款 (No Upper Bound)！
# 真实的社交媒体生态就是由极少数的“超级爆款”主导的。
# 我们仅剔除无人问津的底噪 (最底部的 5%)，确保剩下的视频都真正进入了公共视野。
q_low = df_pure['viewCount'].quantile(0.05)

# 只保留 viewCount 大于底线的视频，上不封顶！！！
df_final = df_pure[df_pure['viewCount'] >= q_low].copy()

print(f"🔪 底噪截断阈值: >= {int(q_low):,} 播放 (上不封顶！)")
print(f"✅ 释放所有顶流爆款后，最终入库靶向视频: {len(df_final)} 条。")

# 💾 Cell 10 - 资产持久化与契约移交 (Asset Persistence & Handoff)
# -----------------------------------------------------------
final_path = OUTPUT_DIR / "01_video_candidates_final.csv"
df_final.to_csv(final_path, index=False, encoding='utf-8-sig')
print(f"\n🚀 [PIPELINE SUCCESS] 最终 2x2 候选集已交付至: {final_path.name}")

# 打印 2x2 矩阵最终分布报表
print("\n📊 核心矩阵数据分布预览：")
summary_df = df_final.groupby('cell_id').agg(
    视频数量=('video_id', 'count'),
    曝光基数中位数=('viewCount', 'median'),
    评论数中位数=('commentCount', 'median'),
    预计可抓取总评论弹药=('commentCount', 'sum')
).reset_index()

display(summary_df)


⚖️ 执行底噪过滤 (Removing Dead-water Videos Only)
🔪 底噪截断阈值: >= 16,577 播放 (上不封顶！)
✅ 释放所有顶流爆款后，最终入库靶向视频: 74 条。

🚀 [PIPELINE SUCCESS] 最终 2x2 候选集已交付至: 01_video_candidates_final.csv

📊 核心矩阵数据分布预览：


,cell_id,视频数量,曝光基数中位数,评论数中位数,预计可抓取总评论弹药
0,non_shein_environment,26,342123.5,618.5,39106
1,non_shein_labor,10,313742.0,697.0,21365
2,shein_environment,26,233211.0,371.0,22611
3,shein_labor,12,244628.0,664.0,13284
